# Modèles de Base — Régression Logistique et Arbre de Décision

Ce notebook entraîne et évalue deux modèles de référence (*baselines*) sur le jeu de données
Kaggle Credit Card Fraud Detection : la **Régression Logistique** et l'**Arbre de Décision**.
Ces modèles simples servent de point de comparaison pour les approches plus avancées.

In [ ]:
import sys, os, warnings
warnings.filterwarnings('ignore')

# Add project root to path
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.models.base_models import create_logistic_regression, create_decision_tree
from src.evaluation.metrics import compute_all_metrics, compute_classification_report
from src.evaluation.visualizer import plot_confusion_matrix, plot_roc_curves

plt.rcParams.update({'font.size': 12, 'figure.dpi': 300, 'font.family': 'serif'})
sns.set_style('whitegrid')

FIGURES_DIR = os.path.join('..', 'reports', 'figures', 'models')
os.makedirs(FIGURES_DIR, exist_ok=True)

print("Imports chargés avec succès.")

In [ ]:
# ── Chargement et préparation des données ──
from src.data.loader import load_creditcard
from src.data.preprocessor import FraudPreprocessor
from src.data.splitter import stratified_split

# Charger le dataset
df = load_creditcard()

# Prétraitement
preprocessor = FraudPreprocessor(scaler_type='standard')
df = preprocessor.handle_missing(df)
df = preprocessor.engineer_features_kaggle(df)

# Séparer features et cible
TARGET = 'Class'
FEATURES = [c for c in df.columns if c != TARGET]

X = df[FEATURES]
y = df[TARGET]

# Normalisation
X_scaled = preprocessor.fit_transform(X)

# Split stratifié train / val / test
X_train, X_val, X_test, y_train, y_val, y_test = stratified_split(X_scaled, y)

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")
print(f"Taux de fraude — Train: {y_train.mean()*100:.3f}%, Test: {y_test.mean()*100:.3f}%")

## Régression Logistique

La régression logistique est un modèle linéaire classique, souvent utilisé comme première
référence. Nous utilisons `class_weight='balanced'` pour compenser le déséquilibre des classes.

In [ ]:
# ── Régression Logistique : entraînement et évaluation ──
lr_model = create_logistic_regression()
lr_model.fit(X_train, y_train)

# Prédictions
y_pred_lr = lr_model.predict(X_test)
y_proba_lr = lr_model.predict_proba(X_test)[:, 1]

# Métriques
metrics_lr = compute_all_metrics(y_test, y_pred_lr, y_proba_lr)
print("=== Régression Logistique — Résultats sur le Test Set ===\n")
print(compute_classification_report(y_test, y_pred_lr))
print(f"AUC-ROC: {metrics_lr['auc_roc']:.4f}")
print(f"AUPRC:   {metrics_lr['auprc']:.4f}")
print(f"F1:      {metrics_lr['f1_score']:.4f}")
print(f"Rappel:  {metrics_lr['recall']:.4f}")

In [ ]:
# ── Matrice de confusion — Régression Logistique ──
plot_confusion_matrix(y_test, y_pred_lr, model_name="Régression Logistique",
                      save_path="models/cm_logistic_regression")
plt.show()

## Arbre de Décision

L'arbre de décision capture des relations non linéaires mais est sensible au sur-apprentissage.
Nous limitons `max_depth=10` pour réduire ce risque.

In [ ]:
# ── Arbre de Décision : entraînement et évaluation ──
dt_model = create_decision_tree()
dt_model.fit(X_train, y_train)

# Prédictions
y_pred_dt = dt_model.predict(X_test)
y_proba_dt = dt_model.predict_proba(X_test)[:, 1]

# Métriques
metrics_dt = compute_all_metrics(y_test, y_pred_dt, y_proba_dt)
print("=== Arbre de Décision — Résultats sur le Test Set ===\n")
print(compute_classification_report(y_test, y_pred_dt))
print(f"AUC-ROC: {metrics_dt['auc_roc']:.4f}")
print(f"AUPRC:   {metrics_dt['auprc']:.4f}")
print(f"F1:      {metrics_dt['f1_score']:.4f}")
print(f"Rappel:  {metrics_dt['recall']:.4f}")

In [ ]:
# ── Matrice de confusion — Arbre de Décision ──
plot_confusion_matrix(y_test, y_pred_dt, model_name="Arbre de Décision",
                      save_path="models/cm_decision_tree")
plt.show()

## Comparaison

Superposition des courbes ROC et tableau récapitulatif des deux modèles de base.

In [ ]:
# ── Courbes ROC superposées ──
roc_models = {
    "Régression Logistique": (y_test, y_proba_lr),
    "Arbre de Décision":     (y_test, y_proba_dt),
}

plot_roc_curves(roc_models, save_path="models/roc_curves_baselines")
plt.show()

print("Figure sauvegardée dans reports/figures/models/roc_curves_baselines.png")

In [ ]:
# ── Tableau comparatif ──
comparison_data = {
    "Modèle": ["Régression Logistique", "Arbre de Décision"],
    "AUC-ROC": [metrics_lr['auc_roc'], metrics_dt['auc_roc']],
    "AUPRC":   [metrics_lr['auprc'], metrics_dt['auprc']],
    "F1-Score": [metrics_lr['f1_score'], metrics_dt['f1_score']],
    "Précision": [metrics_lr['precision'], metrics_dt['precision']],
    "Rappel":   [metrics_lr['recall'], metrics_dt['recall']],
    "Spécificité": [metrics_lr['specificity'], metrics_dt['specificity']],
}

df_comp = pd.DataFrame(comparison_data).set_index("Modèle")
display(df_comp.style.format("{:.4f}").highlight_max(axis=0, color='lightgreen'))

## Observations

1. **AUC-ROC de la Régression Logistique** : Étonnamment correcte grâce à la linéarité des
   features PCA (V1-V28). Le modèle profite du `class_weight='balanced'` pour maintenir un
   rappel décent malgré le déséquilibre extrême.

2. **Arbre de Décision — sur-apprentissage** : L'arbre de décision présente un écart significatif
   entre les performances sur le train et le test, signe de sur-apprentissage. Même avec la
   limitation de profondeur (`max_depth=10`), le modèle mémorise des patterns bruités.

3. **Rappel insuffisant** : Les deux modèles peinent à atteindre un rappel satisfaisant sur la
   classe minoritaire (fraude). C'est une limitation attendue pour des modèles simples face à
   un ratio de déséquilibre de ~578:1.

4. **Conclusion** : Ces baselines confirment la nécessité d'approches plus sophistiquées
   (ensembles, deep learning) pour améliorer la détection de fraude. Ils servent de seuil
   minimal que tout modèle avancé devra dépasser.